In [53]:
"""
APSP: All-Pairs Shortest Paths
"""

'\nAPSP: All-Pairs Shortest Paths\n'

In [45]:
import math

def extend_shortest_paths(L_prev, W, n):
    L_new = [[math.inf] * n for _ in range(n)]
    
    for i in range(n):
        for j in range(n):
            for k in range(n):
                L_new[i][j] = min(L_new[i][j],
                                  L_prev[i][k] + W[k][j])
    return L_new

In [46]:
def slow_apsp(W):
    n = len(W)
    
    # L^(1) = W
    L = W
    
    for _ in range(n - 2):
        L = extend_shortest_paths(L, W, n)
    
    return L

In [47]:
def slow_apsp_with_layers(W):
    n = len(W)
    L = [None] * n
    L[0] = W

    for r in range(1, n):
        L[r] = extend_shortest_paths(L[r-1], W, n)

    return L

In [48]:
def faster_apsp(W):
    n = len(W)
    
    L = W
    m = 1
    
    while m < n - 1:
        L = extend_shortest_paths(L, L, n)  # square it
        m *= 2
    
    return L

In [49]:
def faster_apsp_with_layers(W):
    n = len(W)
    layers = {1: W}
    
    r = 1
    while r < n - 1:
        layers[2*r] = extend_shortest_paths(layers[r], layers[r], n)
        r *= 2
    
    return layers

In [50]:
"""
Slow APSP:
Let's try paths with 1 more edge each time.

Faster APSP:
Let's double how far we can go each time.

Any shortest path has at most n−1 edges, so:

Slow: takes n−1 iterations
Faster: reaches ≥ n−1 using doubling

| Method | Progression  | # Steps  |
| -------| ------------ | -------- |
| Slow   | L¹ → L² → L³ | O(n)     |
| Faster | L¹ → L² → L⁴ | O(log n) |

GRAPH:

0 --1--> 1 --2--> 2
 \--------------5--->

This means:
0→1 costs 1
1→2 costs 2
0→2 costs 5 (direct edge, but not optimal)

INITIAL MATRIX W (L^(1)):
[0   1   5]
[∞   0   2]
[∞   ∞   0]


STEP: W ⊗ W (this is where improvement happens)

We try all "middle nodes k" for each pair (i, j).

Focus on entry (0, 2):

Try k = 0:
0→0 + 0→2 = 0 + 5 = 5

Try k = 1:
0→1 + 1→2 = 1 + 2 = 3 BEST PATH FOUND

Try k = 2:
0→2 + 2→2 = 5 + 0 = 5


RESULTING MATRIX (L^(2)):
[0   1   3]
[∞   0   2]
[∞   ∞   0]


WHAT JUST HAPPENED (INTUITION):
The matrix multiplication is not “multiplying numbers”.
It is testing ALL possible middle stops k.

So instead of only using the direct edge 0→2 = 5,
it discovers a better two-step route:
0 → 1 → 2 = 1 + 2 = 3

This is why the value changes from 5 → 3.

---

WHY THIS ENABLES FAST APSP:

Slow version:
- builds paths like a staircase
- 1 edge → 2 edges → 3 edges → 4 edges

Fast version:
- combines whole blocks of paths
- 1 edge → 2 edges → 4 edges → 8 edges

Because each multiplication already considers ALL middle nodes,
it “jumps” path length instead of stepping one edge at a time.
"""

'\nSlow APSP:\nLet\'s try paths with 1 more edge each time.\n\nFaster APSP:\nLet\'s double how far we can go each time.\n\nAny shortest path has at most n−1 edges, so:\n\nSlow: takes n−1 iterations\nFaster: reaches ≥ n−1 using doubling\n\n| Method | Progression  | # Steps  |\n| -------| ------------ | -------- |\n| Slow   | L¹ → L² → L³ | O(n)     |\n| Faster | L¹ → L² → L⁴ | O(log n) |\n\nGRAPH:\n\n0 --1--> 1 --2--> 2\n \\--------------5--->\n\nThis means:\n0→1 costs 1\n1→2 costs 2\n0→2 costs 5 (direct edge, but not optimal)\n\nINITIAL MATRIX W (L^(1)):\n[0   1   5]\n[∞   0   2]\n[∞   ∞   0]\n\n\nSTEP: W ⊗ W (this is where improvement happens)\n\nWe try all "middle nodes k" for each pair (i, j).\n\nFocus on entry (0, 2):\n\nTry k = 0:\n0→0 + 0→2 = 0 + 5 = 5\n\nTry k = 1:\n0→1 + 1→2 = 1 + 2 = 3   ← BEST PATH FOUND\n\nTry k = 2:\n0→2 + 2→2 = 5 + 0 = 5\n\n\nRESULTING MATRIX (L^(2)):\n[0   1   3]\n[∞   0   2]\n[∞   ∞   0]\n\n\nWHAT JUST HAPPENED (INTUITION):\n\nThe matrix multiplication i

In [52]:
"""
Semiring
switching to the algebra that matches the problem's structure

Standard algebra = how things combine numerically in physics / linear systems
Min-plus algebra = how optimal decisions combine in path problems

Same shape (matrices, multiplication), different meaning underneath.

⊗ behaves like matrix multiplication structurally, but it is a different algebra:
min-plus (tropical) matrix multiplication

normal linear algebra → (+,×)
APSP algebra → (min,+)

(A ⊗ B)[i,j] = min[k](A[i,k] + B[k,j])

| Concept         | Standard linear algebra | APSP (min-plus algebra) |
| --------------- | ----------------------- | ----------------------- |
| addition        | (+)                     | min                     |
| multiplication  | (×)                     | (+)                     |
| identity matrix | 1 on diagonal           | 0 on diagonal           |
| “zero element”  | 0                       | ∞                       |
| matrix product  | sum of products         | min of sums             |
|     L^(0)       | not meaningful here     | identity element        |

Matrix multiplication defined by EXTEND-SHORTEST-PATHS is associative because both:

path concatenation (addition) and
choice of best path (minimum over intermediates)

are operations that can be regrouped without changing the set of all candidate paths.
"""

'\n⊗ behaves like matrix multiplication structurally, but it is a different algebra:\n\nnormal linear algebra → (+,×)\nAPSP algebra → (min,+)\n\n(A ⊗ B)[i,j] = min[k](A[i,k] + B[k,j])\n'

In [54]:
# EXTEND-SHORTEST-PATHS (min-plus) associativity proof

# Define operation:
# (A ⊗ B)[i,j] = min_k (A[i,k] + B[k,j])

# Goal:
# (A ⊗ B) ⊗ C == A ⊗ (B ⊗ C)

# -------------------------------
# LEFT-HAND SIDE
# -------------------------------

# ((A ⊗ B) ⊗ C)[i,j]

# Step 1: expand outer multiplication
LHS = "min_k ( (A ⊗ B)[i,k] + C[k,j] )"

# Step 2: expand (A ⊗ B)
# (A ⊗ B)[i,k] = min_x (A[i,x] + B[x,k])

LHS_expanded = "min_k ( min_x (A[i,x] + B[x,k]) + C[k,j] )"

# Step 3: flatten nested mins
LHS_flat = "min_{k,x} (A[i,x] + B[x,k] + C[k,j])"


# -------------------------------
# RIGHT-HAND SIDE
# -------------------------------

# (A ⊗ (B ⊗ C))[i,j]

# Step 1: expand outer multiplication
RHS = "min_x ( A[i,x] + (B ⊗ C)[x,j] )"

# Step 2: expand (B ⊗ C)
# (B ⊗ C)[x,j] = min_k (B[x,k] + C[k,j])

RHS_expanded = "min_x ( A[i,x] + min_k (B[x,k] + C[k,j]) )"

# Step 3: flatten nested mins
RHS_flat = "min_{x,k} (A[i,x] + B[x,k] + C[k,j])"


# -------------------------------
# FINAL COMPARISON
# -------------------------------

proof = """
LHS = min_{k,x} (A[i,x] + B[x,k] + C[k,j])
RHS = min_{x,k} (A[i,x] + B[x,k] + C[k,j])

Since minimization over all pairs (x,k) is order-independent:
LHS == RHS
⇒ associativity holds
"""

print(proof)


LHS = min_{k,x} (A[i,x] + B[x,k] + C[k,j])
RHS = min_{x,k} (A[i,x] + B[x,k] + C[k,j])

Since minimization over all pairs (x,k) is order-independent:
LHS == RHS
⇒ associativity holds



In [56]:
def initialize(W):
    n = len(W)
    
    L = [row[:] for row in W]
    Pi = [[None] * n for _ in range(n)]

    for i in range(n):
        for j in range(n):
            if i != j and W[i][j] != float('inf'):
                Pi[i][j] = i

    return L, Pi

In [55]:
def extend_shortest_paths(L_prev, Pi_prev, W):
    n = len(L_prev)
    
    L_new = [[float('inf')] * n for _ in range(n)]
    Pi_new = [[None] * n for _ in range(n)]

    for i in range(n):
        for j in range(n):
            # Start with previous best (≤ r-1 edges)
            L_new[i][j] = L_prev[i][j]
            Pi_new[i][j] = Pi_prev[i][j]

            # Try extending via k
            for k in range(n):
                if L_prev[i][k] == float('inf') or W[k][j] == float('inf'):
                    continue

                val = L_prev[i][k] + W[k][j]

                if val < L_new[i][j]:
                    L_new[i][j] = val
                    Pi_new[i][j] = k   # predecessor of j is k

    return L_new, Pi_new

In [57]:
def slow_apsp_w_predecessors(W):
    n = len(W)

    L, Pi = initialize(W)

    for r in range(2, n):
        L, Pi = EXTEND_SHORTEST_PATHS(L, Pi, W)

    return L, Pi

In [58]:
"""
After each matrix squaring step, check whether any diagonal entry L[i,i] is negative.
If so, report that the graph contains a negative-weight cycle.
"""
def faster_apsp_negative_cycle(W):
    n = len(W)

    L = [row[:] for row in W]
    m = 1

    while m < n - 1:
        L_new = [[float('inf')] * n for _ in range(n)]

        for i in range(n):
            for j in range(n):
                for k in range(n):
                    val = L[i][k] + L[k][j]
                    if val < L_new[i][j]:
                        L_new[i][j] = val

        L = L_new
        m *= 2

        for i in range(n):
            if L[i][i] < 0:
                return True

    return False

In [59]:
"""
Computing shortest paths with bounded edges using min-plus powers,
and binary search for the smallest r where a negative diagonal appears.

That r is exactly:
the minimum number of edges in any negative-weight cycle.

What is the shortest cycle I can form if I am only allowed paths up to length r?
it doubles each time, A -> C -> D -> A
r = 2 -> r = 4, finds at 4 then zooms into 3.
"""
def min_length_negative_cycle(W):
    n = len(W)

    # precompute powers L^(1), L^(2), L^(4)
    L_powers = []
    L = [row[:] for row in W]
    L_powers.append(L)

    k = 1
    while k < n:
        L_new = [[float('inf')] * n for _ in range(n)]
        for i in range(n):
            for j in range(n):
                for t in range(n):
                    val = L[i][t] + L[t][j]
                    if val < L_new[i][j]:
                        L_new[i][j] = val

        L = L_new
        L_powers.append(L)
        k *= 2

    def has_negative_cycle(L):
        return any(L[i][i] < 0 for i in range(n))

    result = float('inf')
    L_current = [[float('inf')] * n for _ in range(n)]

    for i in range(n):
        L_current[i][i] = 0

    r = 0

    for i in reversed(range(len(L_powers))):
        candidate = [[float('inf')] * n for _ in range(n)]
        for a in range(n):
            for b in range(n):
                for c in range(n):
                    val = L_current[a][c] + L_powers[i][c][b]
                    if val < candidate[a][b]:
                        candidate[a][b] = val

        if not has_negative_cycle(candidate):
            L_current = candidate
            r += (1 << i)

    return r + 1

In [60]:
"""
Book defines a new nxn matrix in the first loop
but we edit the new matrix in place so its not needed.
"""
def floyd_warshall(W):
    """
    W: adjacency matrix (n x n)
       W[i][j] = weight of edge i→j
       use float('inf') if no edge
       W[i][i] should be 0

    returns:
       D = shortest-path distance matrix
    """
    n = len(W)

    # Copy input matrix so we don't overwrite original
    D = [row[:] for row in W]

    for k in range(n):          # intermediate vertex
        for i in range(n):      # source
            for j in range(n):  # destination
                if D[i][k] + D[k][j] < D[i][j]:
                    D[i][j] = D[i][k] + D[k][j]

    return D

In [61]:
inf = float('inf')

W = [
    [0,   3,   inf, 7],
    [8,   0,   2,   inf],
    [5,   inf, 0,   1],
    [2,   inf, inf, 0]
]

D = floyd_warshall(W)

for row in D:
    print(row)

[0, 3, 5, 6]
[5, 0, 2, 3]
[3, 6, 0, 1]
[2, 5, 7, 0]


In [62]:
def floyd_warshall_with_path(W):
    n = len(W)
    D = [row[:] for row in W]
    Pi = [[None]*n for _ in range(n)]

    # predecessors
    for i in range(n):
        for j in range(n):
            if i != j and W[i][j] != float('inf'):
                Pi[i][j] = i

    for k in range(n):
        for i in range(n):
            for j in range(n):
                if D[i][k] + D[k][j] < D[i][j]:
                    D[i][j] = D[i][k] + D[k][j]
                    Pi[i][j] = Pi[k][j]

    return D, Pi

In [63]:
inf = float('inf')

W = [
    [0,   3,   inf, 7],
    [8,   0,   2,   inf],
    [5,   inf, 0,   1],
    [2,   inf, inf, 0]
]

D = floyd_warshall_with_path(W)

for row in D:
    print(row)

[[0, 3, 5, 6], [5, 0, 2, 3], [3, 6, 0, 1], [2, 5, 7, 0]]
[[None, 0, 1, 2], [3, None, 1, 2], [3, 0, None, 2], [3, 0, 1, None]]


In [64]:
"""
| Problem            | Operation |
| ------------------ | --------- |
| shortest paths     | min +     |
| transitive closure | OR + AND  |
"""
def transitive_closure(G):
    n = len(G)
    T = [row[:] for row in G]

    for i in range(n):
        T[i][i] = 1

    for k in range(n):
        for i in range(n):
            for j in range(n):
                T[i][j] = T[i][j] or (T[i][k] and T[k][j])

    return T

In [65]:
"""
The predecessor pointers always move "backward along shortest paths," and since distances
can't decrease forever, you can't loop so everything forms a tree.

Floyd–Warshall works in-place because each step only introduces one new intermediate vertex,
so updated values remain valid immediately.
"""

'\nThe predecessor pointers always move "backward along shortest paths," and since distances\ncan\'t decrease forever, you can\'t loop so everything forms a tree.\n'

In [67]:
def floyd_warshall_in_place(W):
    n = len(W)
    D = [row[:] for row in W]

    for k in range(n):          
        for i in range(n):
            for j in range(n):
                D[i][j] = min(D[i][j], D[i][k] + D[k][j])

    return D

In [68]:
inf = float('inf')

W = [
    [0,   3,   inf],
    [inf, 0,   1],
    [2,   inf, 0]
]

print(floyd_warshall_in_place(W))

[[0, 3, 4], [3, 0, 1], [2, 5, 0]]


In [69]:
"""
distance table = "what is cheapest way from i to j?"
predecessor table = "what is the first step on that cheapest way?"

So whenever the cheapest way changes, the first step must change too.
"""

'\ndistance table = "what is cheapest way from i to j?"\npredecessor table = "what is the first step on that cheapest way?"\n\nSo whenever the cheapest way changes, the first step must change too.\n'

In [71]:
"""
To detect negative cycles using Floyd–Warshall,
simply check whether any diagonal entry of the final distance matrix is negative.
"""
def floyd_warshall_neagtive_cycles(W):
    n = len(W)
    D = [row[:] for row in W]

    for k in range(n):
        for i in range(n):
            for j in range(n):
                D[i][j] = min(D[i][j], D[i][k] + D[k][j])

    for i in range(n):
        if D[i][i] < 0:
            print("Negative-weight cycle detected")
            break

    return D

In [ ]:
"""
We introduce the intermediate vertex k because finding shortest paths directly from i to j is combinatorially hard,
but breaking paths into "through k" cases gives a clean recursive structure we can actually compute efficiently.

It turns an exponential problem into a polynomial one by:

Before:
all paths
After:
all pairs + one intermediate vertex at a time

So instead of:
enumerate paths

we do:
gradually expand allowed intermediate vertices

Floyd–Warshall is not really "finding paths."

It is:
building the closure of the graph under the operation:

i → k → j
Each k allows us to compose paths.

Think of building routes in a city:

Direct question:
"What is the best route from home to airport?"
too many possibilities

Floyd–Warshall idea:
"Is it better to go via each possible subway station k?"
only n choices per step

Then we recursively improve subroutes.

k grows until it includes every vertex that might be useful, including those on the optimal path.
"""

In [72]:
"""
We achieve O(VE) transitive closure by repeatedly propagating reachability along edges,
in a Bellman–Ford-style relaxation process over the adjacency list representation.
"""
def transitive_closure_VE(G):
    """
    G: adjacency list
       G[u] = list of neighbors of u
    returns: reachability matrix T
    """

    n = len(G)
    T = [[0] * n for _ in range(n)]

    # each node reaches itself
    for i in range(n):
        T[i][i] = 1

    # direct edges
    for u in range(n):
        for v in G[u]:
            T[u][v] = 1

    # repeat V times (like Bellman-Ford style relaxation)
    for _ in range(n):
        for u in range(n):
            for v in range(n):
                if T[u][v]:
                    # propagate u -> neighbors of v
                    for w in G[v]:
                        T[u][w] = 1

    return T

In [74]:
G = [
    [1],    # 0 → 1
    [2],    # 1 → 2
    [],     # 2
    [3]     # 3 → 3 (self-loop)
]
print(transitive_closure_VE(G))
"""
    0 1 2 3
0 [ 1 1 1 0 ]
1 [ 0 1 1 0 ]
2 [ 0 0 1 0 ]
3 [ 0 0 0 1 ]
"""

[[1, 1, 1, 0], [0, 1, 1, 0], [0, 0, 1, 0], [0, 0, 0, 1]]


In [75]:
"""
some notes (refresher):
0 → 1 → 2
↑       ↓
3 ←------
4 → 5
5 → 4

SCC 1:
0, 1, 2, 3
0 → 1 → 2 → 3 → 0 (cycle exists through paths)
everything mutually reachable (via cycles)

SCC 2:
4, 5
4 ↔ 5 (two-way cycle)

{0,1,2,3}   {4,5}

You collapse each SCC into a single node:
[SCC A] → [SCC B]
This always becomes a DAG (no cycles)

They simplify graphs:
cycles disappear
reachability becomes easier
transitive closure becomes easier

SCC = cycle cluster
Tarjan = detect SCCs while DFS runs
Kosaraju = detect SCCs after reversing and reordering
"""

'\nsome notes (refresher):\n0 → 1 → 2\n↑       ↓\n3 ←------\n4 → 5\n5 → 4\n\nSCC 1:\n0, 1, 2, 3\n0 → 1 → 2 → 3 → 0 (cycle exists through paths)\neverything mutually reachable (via cycles)\n\nSCC 2:\n4, 5\n4 ↔ 5 (two-way cycle)\n\n{0,1,2,3}   {4,5}\n\nYou collapse each SCC into a single node:\n[SCC A] → [SCC B]\nThis always becomes a DAG (no cycles)\n\nThey simplify graphs:\ncycles disappear\nreachability becomes easier\ntransitive closure becomes easier\n\nSCC = cycle cluster\nTarjan = detect SCCs while DFS runs\nKosaraju = detect SCCs after reversing and reordering\n'

In [76]:
"""
All-pairs shortest paths
even if edges can be negative
but no negative cycles

Add a fake node s and run Bellman-Ford
Use those results to "fix" edge weights (remove negatives)
Run Dijkstra’s algorithm from every node

Shift all edge weights so nothing is negative → run fast algorithm → shift results back
"""
import heapq

def bellman_ford(graph, V, source):
    dist = {v: float('inf') for v in V}
    dist[source] = 0

    for _ in range(len(V) - 1):
        for u in graph:
            for v, w in graph[u]:
                if dist[u] + w < dist[v]:
                    dist[v] = dist[u] + w

    # negative cycles
    for u in graph:
        for v, w in graph[u]:
            if dist[u] + w < dist[v]:
                return None

    return dist

def dijkstra(graph, source):
    dist = {v: float('inf') for v in graph}
    dist[source] = 0
    pq = [(0, source)]

    while pq:
        d, u = heapq.heappop(pq)
        if d > dist[u]:
            continue

        for v, w in graph[u]:
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
                heapq.heappush(pq, (dist[v], v))

    return dist

def johnson(graph):
    G_prime = {u: list(graph[u]) for u in graph}
    s = 's'
    G_prime[s] = []

    for v in graph:
        G_prime[s].append((v, 0))

    h = bellman_ford(G_prime, G_prime.keys(), s)
    if h is None:
        raise ValueError("negative weight cycle")

    reweighted = {}
    for u in graph:
        reweighted[u] = []
        for v, w in graph[u]:
            new_w = w + h[u] - h[v]
            reweighted[u].append((v, new_w))

    D = {}
    for u in graph:
        dist = dijkstra(reweighted, u)
        D[u] = {}
        for v in graph:
            D[u][v] = dist[v] + h[v] - h[u]

    return D

In [77]:
"""
Johnson is a speed hack for all-pairs shortest paths with negative edges
Convert a hard graph into an easy graph, solve it fast many times, then convert results back.

Bellman-Ford creates a "potential function" that removes negative weights, enabling fast Dijkstra,
and the final step restores the original distance scale.

| Step         | Purpose                                |
| ------------ | -------------------------------------- |
| Bellman-Ford | detect negatives + compute “potential” |
| Reweighting  | remove negative edges                  |
| Dijkstra     | fast shortest paths                    |

What is this "potential h(v)"?

We define:
h(v)=shortest distance from s to v
But more importantly:
h(v) is a potential function (or height) assigned to each node.

Each node has a "height"
Edges get adjusted based on height difference

We transform weights:
ŵ(u,v) = w(u,v) + h(u) − h(v)

This transformation guarantees:
ŵ(u,v) ≥ 0

So now:
Dijkstra becomes valid again

| Algorithm    | Best for                          |
| ------------ | --------------------------------- |
| Dijkstra     | single-source, no negatives       |
| Bellman-Ford | single-source, may have negatives |
| Johnson      | all-pairs + may have negatives    |

But Johnson is NOT always best
Because:
If graph is small → Floyd-Warshall is simpler
If only one source → Bellman-Ford or Dijkstra is better
If no negative edges → just run Dijkstra from each node

========================================

Original graph (has negatives)
        ↓
Add fake node
        ↓
Bellman-Ford → get "potential" h
        ↓
Shift weights → no negatives
        ↓
Run Dijkstra
        ↓
Shift answers back

Johnson’s Algorithm:

Works with:
Adjacency list (most common)
Adjacency matrix (n×n)

Johnson's algorithm relies heavily on:
Bellman-Ford
Dijkstra's algorithm

And those behave differently depending on representation.

With adjacency list + heap:
O((V + E) log V)

With adjacency matrix:
O(V^2)

Adjacency List:
A: [(B,1), (C,4)]
B: [(C,-2), (D,2)]

Only stores existing edges
Fast for iterating neighbors
Ideal for Johnson

Adjacency Matrix
      A    B    C    D
A     0    1    4   ∞
B     ∞    0   -2   2
C     ∞    ∞    0   3
D     ∞    ∞    ∞   0

Stores all possible edges
Uses ∞ for no edge
Easier to visualize matrices

========================================
JOHNSON'S ALGORITHM (4-NODE EXAMPLE)
========================================

INPUT GRAPH G (Adjacency List Form)

graph = {
    'A': [('B', 1), ('C', 4)],
    'B': [('C', -2), ('D', 2)],
    'C': [('D', 3)],
    'D': []
}

----------------------------------------
STEP 0 — ORIGINAL GRAPH G
----------------------------------------

Vertices: {A, B, C, D}

Edges:
A → B (1)
A → C (4)
B → C (-2)
B → D (2)
C → D (3)


----------------------------------------
STEP 1 — ADD NEW NODE s (G')
----------------------------------------

Vertices: {s, A, B, C, D}

Edges:
s → A (0)
s → B (0)
s → C (0)
s → D (0)

A → B (1)
A → C (4)
B → C (-2)
B → D (2)
C → D (3)


----------------------------------------
STEP 2 — BELLMAN-FORD FROM s
----------------------------------------

Compute h(v) = shortest distance from s
Result:

h(s) = 0
h(A) = 0
h(B) = 0
h(C) = -2
h(D) = 0

example
h(C) = -2 because:

Path 1: Direct
s → C = 0

Path 2: Through A
s → A → C
= 0 + 4
= 4

Path 3: Through B
s → B → C
= 0 + (-2)
= -2

Path 4: Longer paths (not needed but Bellman-Ford considers them)
s → A → B → C = 0 + 1 + (-2) = -1
s → B → D → C = 0 + 2 + 3 = 5

0
4
-2  ← smallest
-1
5

Bellman-Ford is NOT one-hop—it explores:
1 edge paths
2 edge paths
3 edge paths
up to V−1 edges
That's how it finds the B → C improvement.

----------------------------------------
STEP 3 — REWEIGHT EDGES
----------------------------------------

Formula:
ŵ(u,v) = w(u,v) + h(u) - h(v)

Apply:

A → B:
1 + 0 - 0 = 1

A → C:
4 + 0 - (-2) = 6

B → C:
-2 + 0 - (-2) = 0

B → D:
2 + 0 - 0 = 2

C → D:
3 + (-2) - 0 = 1


Reweighted Graph Ĝ:

A → B (1)
A → C (6)
B → C (0)
B → D (2)
C → D (1)


----------------------------------------
STEP 4 — DIJKSTRA RESULTS (d̂)
----------------------------------------

Distance Matrix d̂(u,v):

        A    B    C    D
A       0    1    1    2
B       ∞    0    0    1
C       ∞    ∞    0    1
D       ∞    ∞    ∞    0


----------------------------------------
STEP 5 — CONVERT BACK (FINAL D)
----------------------------------------

Formula:
d(u,v) = d̂(u,v) + h(v) - h(u)


From A:
A→A = 0 + 0 - 0 = 0
A→B = 1 + 0 - 0 = 1
A→C = 1 + (-2) - 0 = -1
A→D = 2 + 0 - 0 = 2

From B:
B→A = ∞
B→B = 0 + 0 - 0 = 0
B→C = 0 + (-2) - 0 = -2
B→D = 1 + 0 - 0 = 1

From C:
C→A = ∞
C→B = ∞
C→C = 0 + (-2) - (-2) = 0
C→D = 1 + 0 - (-2) = 3

From D:
D→A = ∞
D→B = ∞
D→C = ∞
D→D = 0 + 0 - 0 = 0


----------------------------------------
FINAL ANSWER MATRIX D
----------------------------------------

        A    B    C    D
A       0    1   -1    2
B       ∞    0   -2    1
C       ∞    ∞    0    3
D       ∞    ∞    ∞    0
"""

'\nJohnson is a speed hack for all-pairs shortest paths with negative edges\nConvert a hard graph into an easy graph, solve it fast many times, then convert results back.\n\nBellman-Ford creates a "potential function" that removes negative weights, enabling fast Dijkstra,\nand the final step restores the original distance scale.\n\n| Step         | Purpose                                |\n| ------------ | -------------------------------------- |\n| Bellman-Ford | detect negatives + compute “potential” |\n| Reweighting  | remove negative edges                  |\n| Dijkstra     | fast shortest paths                    |\n\nWhat is this "potential h(v)"?\n\nWe define:\nh(v)=shortest distance from s to v\nBut more importantly:\nh(v) is a potential function (or height) assigned to each node.\n\nEach node has a "height"\nEdges get adjusted based on height difference\n\nWe transform weights:\nŵ(u,v) = w(u,v) + h(u) − h(v)\n\nThis transformation guarantees:\nŵ(u,v) ≥ 0\n\nSo now:\nDijkstra